# Grover's Search Algorithm

Find one marked item among $N=8$ by amplitude amplification, and watch success probability peak then over-rotate.

**Objectives:**
- Build a phase oracle that flips the sign of a chosen basis state (here `|101>`) using a 3-controlled-Z.
- Build the diffusion operator (the reflection about the uniform superposition).
- Compute the *exact* success probability at each iteration from the state vector.
- Plot success probability vs. iteration count and read off the optimal $\lfloor \tfrac{\pi}{4}\sqrt{N}\rfloor = 2$ iterations.
- See over-rotation: pushing past the peak sends probability back down.

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt
from lib.utils.statevector import statevector

# Use local simulator by default (free, instant)
device = LocalSimulator()

## 1. The problem and the plan

We have $N = 2^n = 8$ items indexed by 3-bit strings, and an oracle that recognizes exactly one of them. Classically, finding it takes up to $N$ checks. Grover finds it in $O(\sqrt{N})$.

The recipe is short:

1. **Superpose** all 8 states with a Hadamard on every qubit. Each amplitude is $1/\sqrt{8}$, so the marked state has probability $1/8$ to begin with.
2. **Repeat** a *Grover iteration* = oracle followed by diffuser. Each iteration is two reflections, which compose into a small rotation of the whole state toward the marked item.
3. **Measure.** After about $\frac{\pi}{4}\sqrt{N}$ iterations the marked amplitude is near 1.

**Convention reminder:** qcsim is *big-endian* — qubit 0 is the leftmost (most-significant) bit of every bitstring. We mark `|101>`, meaning qubit0=1, qubit1=0, qubit2=1. As an integer index that is $101_2 = 5$, so the marked amplitude lives at `statevector(circuit)[5]`.

In [ ]:
n = 3                     # qubits
N = 2 ** n                # search space size = 8
MARKED = "101"            # the item we want to find (big-endian: q0=1, q1=0, q2=1)
marked_index = int(MARKED, 2)   # integer index into the state vector = 5

print(f"Search space N = {N} items")
print(f"Marked item    = |{MARKED}>  ->  state-vector index {marked_index}")
print(f"Initial success probability after one round of H = 1/N = {1/N}")

## 2. A 3-controlled-Z from H + CCNOT + H

Both the oracle and the diffuser need a **CCZ** — a controlled-controlled-Z that flips the sign of `|111>` only and leaves the other seven basis states alone. qcsim has no native CCZ, but there is a standard identity:

$$\text{CCZ} = (I\otimes I\otimes H)\,\text{CCNOT}\,(I\otimes I\otimes H).$$

Conjugating the target of a CCNOT by Hadamards turns the controlled-NOT into a controlled-phase. We sandwich qubit 2 (the target) and control on qubits 0 and 1; by symmetry CCZ does not actually care which qubit is the "target," it phases `|111>` regardless.

In [ ]:
def apply_ccz(circ):
    """Append a controlled-controlled-Z over qubits (0, 1, 2): flips the sign of |111> only."""
    circ.h(2)
    circ.ccnot(0, 1, 2)
    circ.h(2)
    return circ

# Verify CCZ acts as diag(1,1,1,1,1,1,1,-1): only |111> picks up a minus sign.
# Prepare the uniform superposition, apply CCZ, and read the (rescaled) amplitudes.
_check = Circuit()
for q in range(n):
    _check.h(q)
apply_ccz(_check)
_amps = np.real(np.round(statevector(_check) * np.sqrt(N), 6))
print("amplitudes * sqrt(N) after CCZ on uniform state:")
for i in range(N):
    print(f"  |{format(i, '03b')}>: {_amps[i]:+.1f}")

# Assert: every basis state keeps +1 except |111>, which is flipped to -1.
expected = np.ones(N)
expected[int("111", 2)] = -1.0
assert np.allclose(_amps, expected), _amps
print("\nCCZ verified: it phases exactly |111> and nothing else.")

## 3. The phase oracle

The oracle must flip the sign of *our* marked state, `|101>`, not of `|111>`. The trick: surround the CCZ with `X` gates on whichever qubits are `0` in the target. An `X` on a qubit swaps the roles of 0 and 1 there, so the all-ones pattern that CCZ phases lines up with our marked pattern, and the `X` gates undo themselves afterward.

For `|101>` the only `0` is qubit 1, so the oracle is: `X(1)`, `CCZ(0,1,2)`, `X(1)`. The net effect is

$$\ket{x} \longmapsto (-1)^{[x = 101]}\,\ket{x}.$$

In [ ]:
def apply_oracle(circ, marked=MARKED):
    """Phase oracle: flip the sign of the `marked` basis state via an X-sandwiched CCZ."""
    zeros = [q for q, bit in enumerate(marked) if bit == "0"]
    for q in zeros:
        circ.x(q)
    apply_ccz(circ)
    for q in zeros:
        circ.x(q)
    return circ

# Verify the oracle flips ONLY |101>. Apply it to the uniform superposition.
_oc = Circuit()
for q in range(n):
    _oc.h(q)
apply_oracle(_oc)
_oamps = np.real(np.round(statevector(_oc) * np.sqrt(N), 6))
print("amplitudes * sqrt(N) after oracle on uniform state:")
for i in range(N):
    print(f"  |{format(i, '03b')}>: {_oamps[i]:+.1f}")

# Assert: the marked index is -1, everything else is +1.
_expected = np.ones(N)
_expected[marked_index] = -1.0
assert np.allclose(_oamps, _expected), _oamps
print(f"\nOracle verified: it phases exactly |{MARKED}> (index {marked_index}).")

## 4. The diffusion operator

The oracle alone is invisible to measurement — a global pattern of signs does not change any probability. The **diffuser** turns those signs into amplitude. It is the reflection about the uniform superposition $\ket{s}$, written $2\ket{s}\bra{s} - I$, which geometrically reflects every amplitude about their *mean*. States that the oracle pushed below the mean get amplified; the rest shrink.

The standard circuit is: Hadamard all qubits, then reflect about `|000>` (which is `X` all, `CCZ`, `X` all), then Hadamard all qubits again — i.e.

$$D = H^{\otimes n}\,(2\ket{0}\bra{0} - I)\,H^{\otimes n}.$$

In [ ]:
def apply_diffuser(circ):
    """Diffusion = reflection about the uniform superposition: H all, X all, CCZ, X all, H all."""
    for q in range(n):
        circ.h(q)
    for q in range(n):
        circ.x(q)
    apply_ccz(circ)
    for q in range(n):
        circ.x(q)
    for q in range(n):
        circ.h(q)
    return circ

# One full Grover iteration = oracle then diffuser. Build the k=1 circuit and look at it.
def grover_circuit(iterations, marked=MARKED):
    """Uniform superposition, then `iterations` rounds of (oracle, diffuser)."""
    circ = Circuit()
    for q in range(n):
        circ.h(q)
    for _ in range(iterations):
        apply_oracle(circ, marked)
        apply_diffuser(circ)
    return circ

one_iter = grover_circuit(1)
print(one_iter)
print(f"\nqubits = {one_iter.qubit_count}, depth = {one_iter.depth}")

## 5. Exact success probability per iteration

Now the core measurement. For each iteration count $k = 0, 1, \dots, 6$ we build the circuit, read its full state vector, and take $|\text{amplitude of }|101\rangle|^2$ — the exact probability that a measurement would return the marked item. No sampling here: the state vector gives the answer to machine precision, which is what we want for a deterministic assert.

Watch the sequence climb to a peak at $k=2$ and then fall back as the rotation *over-rotates* past the marked state.

In [ ]:
success = []
for k in range(7):
    sv = statevector(grover_circuit(k))
    p = float(np.abs(sv[marked_index]) ** 2)
    success.append(p)
    print(f"iteration {k}: P(|{MARKED}>) = {p:.4f}")

success = np.array(success)

# --- Assert 1: with no iterations, the marked state is just one of 8 equal amplitudes. ---
assert np.isclose(success[0], 1 / N, atol=1e-9), success[0]

# --- Assert 2: the optimal stopping point is k = 2 (the first peak), matching theory. ---
# Grover amplitude after k iters is sin((2k+1)*theta/2) with sin(theta/2) = 1/sqrt(N).
# The FIRST peak is the one you actually stop at; argmax over the practical range 0..5
# lands there. (k=6 is a spurious re-peak from the rotation wrapping around -- see below.)
optimal_k = int(np.argmax(success[:6]))
predicted_k = int(np.floor(np.pi / 4 * np.sqrt(N)))
assert optimal_k == 2, optimal_k
assert optimal_k == predicted_k, (optimal_k, predicted_k)

# --- Assert 3: at the optimum the marked item dominates (probability > 0.9). ---
assert success[2] > 0.9, success[2]

# --- Assert 4: going one step too far over-rotates and lowers the probability. ---
assert success[3] < success[2], (success[3], success[2])

print(f"\nOptimal iterations (first peak) = {optimal_k}  =  floor(pi/4 * sqrt({N})) = {predicted_k}")
print(f"Peak success probability        = {success[2]:.4f}")

## 6. Plot: the climb, the peak, and the over-rotation

Grover is a rotation. The marked amplitude is $\sin\!\big((2k+1)\,\tfrac{\theta}{2}\big)$ with $\sin(\theta/2) = 1/\sqrt{N}$, so success probability traces a $\sin^2$ curve in $k$. The first crest at $k=2$ is where you stop. Keep iterating and the angle sweeps past $90^\circ$, the amplitude *over-rotates*, and probability falls — that dip at $k=4$ is the algorithm having rotated almost all the way back. (It eventually swings up again near $k=6$; that wrap-around re-peak is real but useless in practice, since you would already have stopped at the first peak.)

In [ ]:
theta = 2 * np.arcsin(1 / np.sqrt(N))
ks = np.arange(7)
theory = np.sin((2 * ks + 1) * theta / 2) ** 2

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ks, theory, color="#ff9900", linewidth=1.5, zorder=1,
        label="theory: sin^2((2k+1)*theta/2)")
ax.scatter(ks, success, color="#232f3e", s=60, zorder=2, label="exact (state vector)")
ax.axvline(2, color="gray", linestyle="--", linewidth=1)
ax.annotate("optimal: stop here", xy=(2, success[2]), xytext=(2.6, 0.78),
            arrowprops=dict(arrowstyle="->", color="gray"))
ax.annotate("over-rotated", xy=(4, success[4]), xytext=(3.8, 0.30),
            arrowprops=dict(arrowstyle="->", color="gray"))
ax.set_xlabel("Grover iterations k")
ax.set_ylabel(f"P(measure |{MARKED}>)")
ax.set_title("Grover success probability vs. iteration count (N = 8)")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

# Sanity: the exact points and the analytic curve agree.
assert np.allclose(success, theory, atol=1e-9)
print("Exact state-vector probabilities match the sin^2 rotation curve.")

## 7. Now measure it

The state vector is the ground truth, but a real device only gives you *samples*. Run the optimal 2-iteration circuit on the simulator with shots and histogram the outcomes. The marked string `|101>` should tower over the other seven. We compute the sampled probability from `measurement_counts` directly (qcsim has no `.probability()` result type).

In [ ]:
from lib.utils.visualization import plot_histogram
from lib.utils.results import top_results

shots = 2000
result = device.run(grover_circuit(2), shots=shots).result()
counts = result.measurement_counts

print("Top measured outcomes:")
for bitstring, c in top_results(counts, n=4):
    print(f"  |{bitstring}>: {c:5d}  ({c / shots:.3f})")

sampled_p = counts[MARKED] / shots
print(f"\nSampled P(|{MARKED}>) = {sampled_p:.3f}   (exact was {success[2]:.3f})")

# Illustrative only -- this is a wide, lenient check on a RANDOM sample, never a tight assert.
assert sampled_p > 0.8, sampled_p

fig = plot_histogram(counts, title=f"Grover, 2 iterations, {shots} shots (marked |{MARKED}>)")
plt.show()

## Exercises

Three exercises that push on the machinery you just built -- a different
target, two targets at once, and a bigger search space. Each has tiered hints
(expand only what you need) and a check cell that tells you when you have it.
Every check reads an exact state vector, so the results are deterministic.

### Exercise 1 — Mark a different item

Grover does not care *which* item you hunt for: the geometry is identical no
matter where the needle sits. Point the oracle at a different 3-bit string and
confirm the search still concentrates on it after the same optimal 2
iterations.

Define `alt_marked`: any 3-bit string other than the notebook's `|101>`.
Define `alt_probs`: the length-8 probability distribution over the basis states
after running Grover on `alt_marked` for 2 iterations -- the squared magnitudes
of the exact state vector.

<details><summary>Hint 1 — nudge</summary>

Every helper above already takes a `marked` argument, and nothing about the
circuit changes when you swap the target string. So what is the *one* input you
change -- and where in the state vector does the probability of a given basis
state live?

</details>
<details><summary>Hint 2 — approach</summary>

Build the circuit with `grover_circuit(2, marked=alt_marked)`, take its
`statevector(...)`, and square the magnitudes with `np.abs(...) ** 2` to turn
amplitudes into probabilities. The integer index of your marked string is
`int(alt_marked, 2)` -- that is the entry that should dominate the
distribution.

</details>

In [ ]:
# Exercise 1: mark a different 3-bit item and confirm Grover still finds it.
# Define: alt_marked -- a 3-bit string other than "101"; and
#         alt_probs  -- the length-8 probability distribution after 2 iterations.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert (
        isinstance(alt_marked, str) and len(alt_marked) == 3 and set(alt_marked) <= {"0", "1"}
    ), "alt_marked should be a 3-bit string like '011'"
    assert alt_marked != MARKED, "choose a DIFFERENT item from the notebook's |101>"
    _p = np.asarray(alt_probs, dtype=float)
    assert _p.shape == (N,), "alt_probs should hold one probability per basis state (length 8)"
    assert np.isclose(_p.sum(), 1.0, atol=1e-6), "a probability distribution should sum to 1"
    _target = int(alt_marked, 2)
    assert int(np.argmax(_p)) == _target, (
        "after the search the most probable outcome should be your NEW marked item"
    )
    assert _p[_target] > 0.9, (
        "at 2 iterations the marked item should carry more than 90% of the probability"
    )

### Exercise 2 — Two marked items

When $M$ of the $N$ items are marked, the two reflections sweep the state
around faster, so the optimal iteration count drops to
$\lfloor \tfrac{\pi}{4}\sqrt{N/M}\rfloor$. Mark *two* items at once and watch
the search peak sooner.

Define `two_marked`: a list of exactly two distinct 3-bit strings (for instance
`"101"` and `"010"`). Define `two_optimal_k`: the optimal iteration count for
$M = 2$. Define `two_probs`: the length-8 probability distribution after running
a two-item Grover for `two_optimal_k` iterations.

<details><summary>Hint 1 — nudge</summary>

A Grover iteration is oracle-then-diffuser. Marking two items means the oracle
flips the sign of *both* before the single shared diffuser runs -- the diffuser
itself is unchanged. With $M = 2$ and $N = 8$, what does
$\lfloor \tfrac{\pi}{4}\sqrt{N/M}\rfloor$ evaluate to?

</details>
<details><summary>Hint 2 — approach</summary>

Write a circuit like `grover_circuit`, but inside each iteration call
`apply_oracle(circ, m)` once for *each* marked string before the single
`apply_diffuser(circ)`. Run it for `two_optimal_k` iterations (read that
straight off the formula, or scan `k` and take the first peak), then
`statevector(...)` and square the magnitudes. Both marked indices --
`int(s, 2)` for each string -- should share almost all of the probability.

</details>

In [ ]:
# Exercise 2: mark two items at once and find the earlier optimal iteration count.
# Define: two_marked    -- a list of two distinct 3-bit strings;
#         two_optimal_k -- the optimal iteration count for M=2;
#         two_probs     -- the length-8 probability distribution at that optimum.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    _items = list(two_marked)
    assert len(_items) == 2 and len(set(_items)) == 2, "mark exactly two DISTINCT items"
    assert all(
        isinstance(s, str) and len(s) == 3 and set(s) <= {"0", "1"} for s in _items
    ), "each marked item should be a 3-bit string"
    assert int(two_optimal_k) == 1, (
        "with M=2 marks the optimum drops to floor(pi/4*sqrt(N/M)) = 1 iteration"
    )
    _p = np.asarray(two_probs, dtype=float)
    assert _p.shape == (N,), "two_probs should hold one probability per basis state (length 8)"
    assert np.isclose(_p.sum(), 1.0, atol=1e-6), "a probability distribution should sum to 1"
    _idxs = [int(s, 2) for s in _items]
    assert sum(_p[i] for i in _idxs) > 0.95, (
        "at the optimum nearly all the probability should sit on the two marked items together"
    )
    _other = max(p for j, p in enumerate(_p) if j not in _idxs)
    assert _p[_idxs[0]] > _other and _p[_idxs[1]] > _other, (
        "each marked item should be more likely than every unmarked one"
    )

### Exercise 3 — Scale to $n = 4$

Grover's whole payoff is the $\sqrt{N}$ scaling of where you stop. Rebuild the
experiment for $n = 4$ qubits ($N = 16$) and read the optimal iteration count
off the exact success curve; it should land at
$\lfloor \tfrac{\pi}{4}\sqrt{16}\rfloor = 3$.

Define `marked4`: a 4-bit target string. Define `success4`: the exact success
probability of `marked4` at each iteration count $k = 0, 1, \dots, 5$. Define
`optimal_k_4`: the iteration count where that probability first peaks.

<details><summary>Hint 1 — nudge</summary>

The recipe does not change -- superpose, repeat (oracle, diffuser), read the
marked amplitude. Only one primitive grows: the phase flip now depends on
*four* qubits, so the two-controlled `CCZ` becomes a three-controlled `Z`. The
X-sandwich oracle and the reflect-about-`|0000>` diffuser are the same
patterns, one qubit wider.

</details>
<details><summary>Hint 2 — approach</summary>

The one new piece is a three-controlled `Z` (a phase flip on `|1111>`). A clean
way to get it is the compute-uncompute **ancilla** trick: `ccnot` the first two
controls into a spare qubit, `ccnot` the third control with that spare onto the
target, then repeat the first `ccnot` to reset the spare back to `|0>` -- and
wrap the target in `h` gates to turn the NOT into a `Z`. Two qcsim notes if you
take the ancilla route: qcsim drops qubits a circuit never touches, so keep the
ancilla in the register (an `i()` on it works), and the extra qubit shifts where
the marked amplitude sits in the state vector. Then build `oracle`, `diffuser`,
and a `grover` loop for four data qubits just as the notebook did for three, and
record `abs(statevector(...)[marked index]) ** 2` at each `k`.

</details>

In [ ]:
# Exercise 3: rebuild Grover for n=4 (N=16) and locate the optimal iteration count.
# Define: marked4     -- a 4-bit target string;
#         success4    -- exact P(marked4) for k = 0, 1, ..., 5;
#         optimal_k_4 -- the k where success4 first peaks (expect 3).

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert (
        isinstance(marked4, str) and len(marked4) == 4 and set(marked4) <= {"0", "1"}
    ), "marked4 should be a 4-bit string"
    _p4 = np.asarray(success4, dtype=float)
    assert _p4.shape[0] >= 6, "record the success probability for k = 0, 1, ..., 5"
    assert np.all((_p4 >= -1e-9) & (_p4 <= 1 + 1e-9)), "probabilities live in [0, 1]"
    assert np.isclose(_p4[0], 1 / 16, atol=1e-6), (
        "with no iterations the marked item still has probability 1/N = 1/16"
    )
    assert int(optimal_k_4) == 3, (
        "for N=16 the first peak is at floor(pi/4*sqrt(16)) = 3 iterations"
    )
    assert int(np.argmax(_p4[:6])) == 3, "the exact probabilities should peak at k=3"
    assert _p4[3] > 0.9, "at the optimum the marked item should dominate (P > 0.9)"
    assert _p4[4] < _p4[3], (
        "one step past the peak over-rotates and the probability drops"
    )

## Summary

- **Two reflections per iteration.** The oracle flips the sign of the marked state; the diffuser reflects every amplitude about their mean. Composed, they rotate the state toward the marked item.
- **CCZ from H + CCNOT + H** gives the multi-controlled phase both pieces need; the oracle X-sandwiches it onto the chosen string, the diffuser onto `|000>`.
- **Exact probabilities come from the state vector**, not from shots — that is what makes the per-iteration asserts deterministic. Sampling is for the final "now measure it" picture only.
- **Knowing when to stop is part of the algorithm.** For $N=8$ the peak is at $\lfloor \tfrac{\pi}{4}\sqrt{8}\rfloor = 2$ iterations ($P > 0.94$); push past it and the state *over-rotates* and probability falls back.

**Next:** [`03-qft.ipynb`](03-qft.ipynb) — build the Quantum Fourier Transform from Hadamards and controlled phase rotations, the interference engine behind phase estimation and Shor's algorithm.